# BIOT 6900 · Module 8 — Single-Cell Transcriptomics
## Real-data notebook (2 of 3): integrating COVID-19 PBMCs across donors

This is a **run-and-read** worked example. There is no code for you to write. You run it top to bottom and read the explanation above each cell.

The toy notebook used one clean sample. Real data almost never is: it comes from **many donors**, measured on different days and instruments, so the first thing you see in the data is often the *batch*, not the biology. This notebook shows that problem and the standard fix.

- **Dataset:** COVID-19 peripheral blood cells pulled live from the **CZ CELLxGENE Census**. Real human data, many donors.
- **New ideas:** batch effects, **batch integration with scVI** (a GPU step), a fair comparison of integration methods, and the honest limits of "AI-powered" annotation (LO4).
- **Maps to:** LO1, LO2 (integrate multimodal/multi-batch data), LO3, LO4 (evaluate model performance and limitations).

> Toy (done) → **this notebook** → the graded assignment, where you run the same workflow on your own Part I disease by changing one line of the Census query.

## Running this on Explorer (read first)

One step in this notebook, training **scVI**, wants a GPU. The easiest way to get one is Northeastern's **Open OnDemand** web portal. No terminal, no Slurm scripts.

1. Go to **rc.northeastern.edu/ood** and sign in with your Northeastern account.
2. **Interactive Apps → JupyterLab**.
3. In the launch form:
   - **Partition:** select **`courses-gpu`** (our class reservation).
   - **Number of GPUs:** `1`.
   - **Memory:** 32 GB is plenty. **Time:** 3 hours.
4. Click **Launch**, wait for the session to start, then **Open JupyterLab**.
5. Open this notebook, choose the course kernel, and **Run → Run All Cells**.

Notes:
- Only the scVI cell needs the GPU. Everything else runs on CPU.
- If you land on a node without a GPU, the notebook still finishes; scVI just runs slower on CPU and prints a warning.
- You do not install anything on your laptop. The course environment on Explorer already has the packages below.

### Packages

This module needs, on top of the toy's requirements: `cellxgene-census`, `scvi-tools` (which brings in PyTorch), and `harmonypy`. On Explorer's course environment these are already installed. If you are running elsewhere and an import fails, use the install cell once, then restart the kernel.

**Python version:** `cellxgene-census` needs Python 3.13 or earlier (its `tiledbsoma` dependency has no 3.14 wheel yet). The course environment is already on a supported version.

In [ ]:
# Run once only if an import below fails, then restart the kernel.
# Requires Python <= 3.13 (cellxgene-census / tiledbsoma has no 3.14 wheel yet).
# !pip install "cellxgene-census" "scvi-tools" harmonypy scanpy leidenalg igraph

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white")

# Is a GPU attached? scVI will use it if so.
try:
    import torch
    GPU = torch.cuda.is_available()
    print("PyTorch sees a GPU:" , GPU, "|", torch.cuda.get_device_name(0) if GPU else "running on CPU")
except Exception as e:
    GPU = False
    print("PyTorch not importable yet:", e)
print("scanpy", sc.__version__)

## Step 1 · Pull real data from the CZ CELLxGENE Census

The Census is a standardized, versioned collection of public single-cell data. We ask it for **COVID-19 cells from blood**, in one query. To keep the worked example fast we sample **10 whole donors** (2,000 cells each) and download only those. We sample whole donors rather than scattered cells on purpose: the later batch-correction and batch-aware gene-selection steps need enough cells per donor to work.

The returned object carries real metadata: which **donor** each cell came from, the **assay**, and a curated **cell_type** label we can use later as a reference.

In [ ]:
import cellxgene_census

census = cellxgene_census.open_soma(census_version="stable")

# List matching cells (metadata only), then subsample for a fast, in-memory worked example.
obs = cellxgene_census.get_obs(
    census, "Homo sapiens",
    value_filter="disease == 'COVID-19' and tissue_general == 'blood' and is_primary_data == True",
    column_names=["soma_joinid", "donor_id", "assay", "cell_type"],
)
print(f"matching cells in the Census: {len(obs):,}")

# Sample whole donors, not random cells. Random cells would scatter across ~500 donors
# with ~30 cells each: too few per batch for batch-aware HVG selection or integration.
rng = np.random.default_rng(0)
cells_per_donor = 2000
donor_sizes = obs["donor_id"].value_counts()
big_donors = donor_sizes[donor_sizes >= cells_per_donor].index.to_numpy()
donors = rng.choice(big_donors, size=min(10, len(big_donors)), replace=False)
sel = np.concatenate([
    rng.choice(obs.loc[obs["donor_id"] == d, "soma_joinid"].to_numpy(),
               size=cells_per_donor, replace=False)
    for d in donors
])
print(f"sampled {len(sel):,} cells from {len(donors)} donors")

adata = cellxgene_census.get_anndata(
    census, "Homo sapiens",
    obs_coords=np.sort(sel),
    obs_column_names=["donor_id", "assay", "cell_type", "disease", "sex"],
    var_column_names=["feature_id", "feature_name"],
)
census.close()

# Use human-readable gene symbols as the index.
adata.var["ensembl_id"] = adata.var_names
adata.var_names = adata.var["feature_name"].astype(str)
adata.var_names_make_unique()
print(adata)

Look at where these cells come from. Real data is a mix of donors and, often, assays. That mixture is the batch structure we have to deal with.

In [ ]:
print("donors:", adata.obs["donor_id"].nunique())
print(adata.obs["assay"].value_counts())
print("\ncurated cell types (reference labels from the Census):")
print(adata.obs["cell_type"].value_counts().head(12))

# The Census labels are very fine-grained (many monocyte and dendritic subtypes).
# Collapse them to broad types for legible plots and meaningful biology scores.
def broad_type(ct):
    ct = str(ct).lower()
    if "cd4" in ct or "cd8" in ct or "t cell" in ct or "t-cell" in ct or "thymocyte" in ct: return "T"
    if "plasma" in ct or "b cell" in ct or "b-cell" in ct: return "B"
    if "natural killer" in ct or "innate lymphoid" in ct: return "NK"
    if "monocyte" in ct: return "Monocyte"
    if "dendritic" in ct: return "DC"
    if "macrophage" in ct: return "Macrophage"
    if "platelet" in ct or "megakaryocyte" in ct: return "Platelet"
    if "erythro" in ct: return "Erythroid"
    if "progenitor" in ct or "stem cell" in ct or "hematopoietic" in ct: return "Progenitor"
    return "Other"

adata.obs["cell_type_broad"] = adata.obs["cell_type"].map(broad_type).astype("category")
print("\nbroad cell types:")
print(adata.obs["cell_type_broad"].value_counts())

## Step 2 · The standard pipeline (recap from the toy)

Same steps you already know: keep the raw counts aside, QC, normalize, pick highly variable genes. We store the full log-normalized matrix in `adata.raw` so every gene is available for later marker plots, then restrict the working matrix to HVGs.

In [ ]:
# Basic gene/cell filtering
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_genes=200)

# Keep raw counts in a layer (scVI needs them)
adata.layers["counts"] = adata.X.copy()

# QC
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
adata = adata[(adata.obs.n_genes_by_counts < 5000) & (adata.obs.pct_counts_mt < 15)].copy()
print(f"cells after QC: {adata.n_obs:,}")

# Normalize + log, then keep the full matrix for marker plots
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

# Highly variable genes, selected with batch awareness
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3",
                            layer="counts", batch_key="donor_id")
adata = adata[:, adata.var.highly_variable].copy()
print(f"highly variable genes kept: {adata.n_vars:,}")

## Step 3 · The batch problem, made visible

Run the familiar PCA → neighbors → UMAP → Leiden with **no integration**, then color the map two ways: by **donor** and by the Census **cell type**.

If the biology dominated, each cell type would form one group regardless of donor. Watch for the opposite: cells splitting by donor. That is the batch effect.

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")

sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep="X_pca")
sc.tl.umap(adata)
adata.obsm["X_umap_unint"] = adata.obsm["X_umap"].copy()

sc.pl.embedding(adata, basis="umap_unint", color=["donor_id", "cell_type_broad"],
                wspace=0.5, title=["No integration: by donor", "No integration: by cell type"])

## Step 4 · Integrate with scVI (the GPU step)

**scVI** is a probabilistic deep-learning model. It learns a low-dimensional latent space from the raw counts while explicitly accounting for the donor each cell came from, so the embedding reflects biology rather than batch. This is the step that benefits from a GPU.

It trains on the **raw counts** (the `counts` layer), told which column is the batch.

In [ ]:
import scvi

scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="donor_id")
vae = scvi.model.SCVI(adata, n_latent=30)
vae.train(max_epochs=(200 if GPU else 60), accelerator=("gpu" if GPU else "cpu"))

adata.obsm["X_scVI"] = vae.get_latent_representation()

sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_scVI")
sc.tl.umap(adata)
adata.obsm["X_umap_scvi"] = adata.obsm["X_umap"].copy()
sc.tl.leiden(adata, resolution=0.5, key_added="leiden_scvi")

sc.pl.embedding(adata, basis="umap_scvi", color=["donor_id", "cell_type_broad"],
                wspace=0.5, title=["scVI: by donor", "scVI: by cell type"])

Compare this map with the unintegrated one. After scVI, donors should be **mixed within** each biological group, while the cell types stay separated. That is what good integration looks like.

## Step 5 · A fair baseline: Harmony

scVI is not the only option, and it is not automatically the best. **Harmony** is a fast, non-deep-learning method that corrects the PCA embedding directly. We run it as an honest comparison.

In [ ]:
import harmonypy

# Call harmonypy directly. scanpy's harmony_integrate wrapper transposes the result,
# which breaks with current harmonypy releases that already return cells x PCs.
ho = harmonypy.run_harmony(adata.obsm["X_pca"].astype(np.float64), adata.obs, "donor_id")
Z = np.asarray(ho.Z_corr)
adata.obsm["X_pca_harmony"] = Z if Z.shape[0] == adata.n_obs else Z.T

sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca_harmony")
sc.tl.umap(adata)
adata.obsm["X_umap_harmony"] = adata.obsm["X_umap"].copy()

sc.pl.embedding(adata, basis="umap_harmony", color=["donor_id", "cell_type_broad"],
                wspace=0.5, title=["Harmony: by donor", "Harmony: by cell type"])

## Step 6 · Score the methods, and the AI caveat (LO4)

Two things make an integration good: it **mixes batches** (donors blend together) and it **preserves biology** (cell types stay apart). We put a rough number on each, using silhouette scores on a subsample:

- **Batch silhouette** (by donor): *lower is better* — donors are well mixed.
- **Cell-type silhouette** (by label): *higher is better* — biology is preserved.

These are quick proxies for teaching; a real benchmark would use a dedicated toolkit such as `scib-metrics`.

In [ ]:
from sklearn.metrics import silhouette_score

idx = rng.choice(adata.n_obs, size=min(5000, adata.n_obs), replace=False)
reps = {"No integration": "X_pca", "Harmony": "X_pca_harmony", "scVI": "X_scVI"}
rows = []
for name, key in reps.items():
    emb = adata.obsm[key][idx]
    batch_sil = silhouette_score(emb, adata.obs["donor_id"].values[idx])
    bio_sil   = silhouette_score(emb, adata.obs["cell_type_broad"].values[idx])
    rows.append({"method": name, "batch silhouette (lower=better)": round(batch_sil, 3),
                 "cell-type silhouette (higher=better)": round(bio_sil, 3)})
print(pd.DataFrame(rows).to_string(index=False))

**Read the table.** The clear, reliable signal is the **batch silhouette**: both Harmony and scVI pull it down relative to no integration, meaning donors are better mixed. The **cell-type silhouette** (computed on the broad types) is the biology-preservation check: integration should not collapse the cell types together. These are rough proxies, and PBMC types already overlap a fair amount, so do not over-read small differences. The headline still holds: a fast classical method (Harmony) is competitive with the deep-learning one (scVI). More machinery is not automatically more biology.

**The AI caveat, extended.** The same lesson holds one level up, for the single-cell *foundation models* (scGPT, Geneformer) you hear about. In rigorous zero-shot benchmarks they frequently **underperform** exactly these simple baselines on cell-type clustering and batch integration (Kedzierska et al., *Genome Biology*, 2025). We do not run them here, both because they are heavy to deploy and because the evidence says they would not win this contest. That is the course thesis made concrete: **AI accelerates the workflow; your job is to catch where it is confidently wrong.**

## Step 7 · Annotate on the integrated space

The Census already ships a curated `cell_type` for each cell, which is the **reference / atlas** route to annotation. We use it directly and sanity-check it against canonical markers on the scVI map. (In the graded notebook you will annotate clusters yourself, marker by marker.)

In [ ]:
sc.pl.embedding(adata, basis="umap_scvi", color="cell_type_broad",
                title="scVI space, cell types (broad)")

markers = ["CD3D", "MS4A1", "CD14", "FCGR3A", "GNLY", "PPBP"]
markers = [g for g in markers if g in adata.raw.var_names]
sc.pl.embedding(adata, basis="umap_scvi", color=markers, use_raw=True, ncols=3)

## Step 8 · COVID biology: which cells mount the interferon response?

A hallmark of viral infection is the **interferon-stimulated gene (ISG)** program. Rather than a single tissue average, single-cell data shows us *which* cell types switch it on. We score each cell for a small ISG set and map it.

In [ ]:
isg = [g for g in ["ISG15", "IFI6", "MX1", "IFITM3", "OAS1", "STAT1"] if g in adata.raw.var_names]
sc.tl.score_genes(adata, isg, score_name="ISG_score", use_raw=True)

sc.pl.embedding(adata, basis="umap_scvi", color=["ISG_score", "cell_type_broad"], wspace=0.5,
                title=["Interferon-response score", "Cell type"])

print("Mean ISG score by cell type (top responders):")
print(adata.obs.groupby("cell_type_broad", observed=True)["ISG_score"].mean().sort_values(ascending=False).head(8))

## Step 9 · The payoff: which cells express your target?

Exactly as in the toy, but now on integrated, multi-donor, real disease data. Pick a gene and read off the cell types that carry it. In the graded assignment this gene is **your own Part I target**.

In [ ]:
target = "CD14"  # swap for your Part I target in the graded notebook
if target in adata.raw.var_names:
    sc.pl.embedding(adata, basis="umap_scvi", color=target, use_raw=True,
                    title=f"{target} expression across cell types")
else:
    print(target, "not found; pick a gene present in adata.raw.var_names")

## What you just saw

- Real data arrives **batched by donor**, and without integration the batch can masquerade as biology.
- **scVI** (GPU) and **Harmony** (CPU) both fix it; scored fairly, the simple method is competitive, and the hyped foundation models do not win this contest.
- Curated **reference labels** plus **marker checks** annotate the integrated map.
- Single-cell resolution shows *which* cell types mount the COVID interferon response, and which express any target gene.

**Next: the graded assignment.** You will run this workflow on your own Part I disease. The only real change is the Census query, for example swapping `disease == 'COVID-19'` for your disease, and choosing the right tissue. Everything downstream is the same.